## **esm-2 representation**

#### -- this doc aims to use the esm-2 model to represent the seq of antibodies & antigens, and train the Bi-LSTM model to learn the relationship between seq representations and the interface.

#### **1. loading the training dataset**

In [1]:
import pandas as pd

In [2]:
import os
from pathlib import Path

# Resolve repo root regardless of execution cwd.
def find_repo_root(start: Path) -> Path:
    for path in [start] + list(start.parents):
        if (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find repo root with data/")

repo_root = find_repo_root(Path.cwd())

train_csv_arg = os.getenv("EPP_TRAIN_CSV", "data/train.csv")
train_data_path = Path(train_csv_arg)
if not train_data_path.is_absolute():
    train_data_path = repo_root / train_data_path
if not train_data_path.exists():
    raise FileNotFoundError(f"Train CSV not found: {train_data_path}")

output_dir_arg = os.getenv("EPP_OUTPUT_DIR", "data")
output_dir = Path(output_dir_arg)
if not output_dir.is_absolute():
    output_dir = repo_root / output_dir
output_dir.mkdir(parents=True, exist_ok=True)

train_data = pd.read_csv(train_data_path, sep=",")

#### **2. representing data feature using esm-2**

In [3]:
import torch
import esm

In [4]:
# Load ESM-2 model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
model = model.to(device)
model.eval()
batch_converter = alphabet.get_batch_converter()
print(f"Using device: {device}")

Using device: cuda


In [5]:
# 2. input data
pdb_id = train_data["pdbid"].astype(str)
ab_seq = train_data["abseq"].astype(str)
ag_seq = train_data["agseq"].astype(str)

data_train_ab = list(zip(pdb_id + "_" + ab_seq, ab_seq))
data_train_ag = list(zip(pdb_id + "_" + ag_seq, ag_seq))

# data_train = data_train_ab + data_train_ag
data_train = data_train_ag

print(type(data_train))
print(len(data_train))
print(data_train[0])

df = pd.DataFrame(data_train, columns=["protein_name", "seq"])
output_path = output_dir / "esm-rep-data_ag.csv"
df.to_csv(output_path, index=False)

<class 'list'>
1206
('7DF1_D_7DF1_G_HGAGNLAGLKGRLDYLSSLKVKGLVLGPIHKNQKDDVAQTDLLQIDPNFGSKEDFDSLLQSAKKKSIRVILDLTPNYRGENSWFSTQVDTVATKVKDALEFWLQAGVDGFQVRDIENLKDASSFLAEWQNITKGFSEDRLLIAGTNSSDLQQILSLLESNKDLLLTSSYLSDSTKSLVTQYLNATGNRWCSWSLS', 'HGAGNLAGLKGRLDYLSSLKVKGLVLGPIHKNQKDDVAQTDLLQIDPNFGSKEDFDSLLQSAKKKSIRVILDLTPNYRGENSWFSTQVDTVATKVKDALEFWLQAGVDGFQVRDIENLKDASSFLAEWQNITKGFSEDRLLIAGTNSSDLQQILSLLESNKDLLLTSSYLSDSTKSLVTQYLNATGNRWCSWSLS')


In [6]:
# Extract per-residue representations fro each seq
representations_list = []
for i in range(len(data_train)):
    data = [data_train[i]]
    batch_labels, batch_strs, batch_tokens = batch_converter(data)
    batch_tokens = batch_tokens.to(device)
    batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)

    # Extract per-residue representations (on CPU) 
    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[33], return_contacts=True)
    token_representations = results["representations"][33].cpu()
    representations_list.append(token_representations[0][1:-1])

In [7]:
# save representations data
max_cols = max([tensor.size(0) for tensor in representations_list])
print('the max length of protein seq is: ', max_cols)
for i, tensor in enumerate(representations_list):
    if tensor.size(0) < max_cols:
        padding = torch.zeros(max_cols - tensor.size(0), tensor.size(1))
        representations_list[i] = torch.cat([tensor, padding], dim=0)

representations_tensor = torch.stack(representations_list)

output_path = output_dir / "traindata_esm_ag.pt"
torch.save(representations_tensor, output_path)
print('successfully save the representation data of antigen sequences!')

the max length of protein seq is:  250


successfully save the representation data of antigen sequences!


In [8]:
# Extract per-residue representations fro each seq
representations_list = []
for i in range(len(data_train_ab)):
    data = [data_train_ab[i]]
    batch_labels, batch_strs, batch_tokens = batch_converter(data)
    batch_tokens = batch_tokens.to(device)
    batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)

    # Extract per-residue representations (on CPU) 
    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[33], return_contacts=True)
    token_representations = results["representations"][33].cpu()
    representations_list.append(token_representations[0][1:-1])

In [9]:
# save representations data
max_cols = max([tensor.size(0) for tensor in representations_list])
for i, tensor in enumerate(representations_list):
    if tensor.size(0) < max_cols:
        padding = torch.zeros(max_cols - tensor.size(0), tensor.size(1))
        representations_list[i] = torch.cat([tensor, padding], dim=0)

representations_tensor = torch.stack(representations_list)

output_path = output_dir / "traindata_esm_ab.pt"
torch.save(representations_tensor, output_path)
print('successfully save the representation data of antibody sequences!')

successfully save the representation data of antibody sequences!
